# Notebook 04: Drug Discovery Agent — ChEMBL & ClinicalTrials.gov

**CABS AI Productivity Series**
Workshop: *LangChain, LangGraph & Local LLM Deployment: Building AI Agent Systems That Keep Your Data Safe*

---

## What This Notebook Covers

In Notebook 02 we built an agent that queries UniProt and AlphaFold to look up gene biology.
Now we apply the **same agent pattern** to drug discovery questions:

| Tool | Database | Answers questions like… |
|------|----------|------------------------|
| `search_drug_chembl` | ChEMBL (EMBL-EBI) | What drugs target EGFR? What is erlotinib's molecular weight? |
| `get_drug_mechanism` | ChEMBL mechanism API | How does imatinib work? What is its primary target? |
| `search_clinical_trials` | ClinicalTrials.gov v2 | What trials are ongoing for KRAS inhibitors? How many Phase 3 trials exist for PD-1? |

**The architecture is identical to Notebook 02.** The only difference is the domain of the tools.

---

## Data Privacy Note

| What goes where | Cloud (this notebook) | Local option (Notebook 03 pattern) |
|---|---|---|
| Your question | Sent to Google Gemini | Stays on your machine |
| Drug/target names in API calls | Sent to ChEMBL/ClinicalTrials (public databases) | Same — these are public APIs |
| Retrieved data | Processed by Gemini | Processed locally |

If your question reveals an unpublished target or compound, consider running locally with Ollama.


# Setup: Get Your Free Gemini API Key

1. Go to [aistudio.google.com](https://aistudio.google.com)
2. Sign in with your Google account → Accept Terms of Service
3. Left sidebar → **Get API Key** → **Create API key**
4. Copy the key (starts with `AIza...`)

No credit card needed.


In [ ]:
# ============================================================
# STEP 1: Install required packages
# ============================================================
# langchain              - framework for building LLM applications
# langchain-google-genai - connects LangChain to Google Gemini
# langgraph              - for building agent execution graphs
# requests               - for calling external APIs (ChEMBL, ClinicalTrials)

!pip install -q langchain langchain-google-genai langgraph requests
print("✅ Packages installed!")


In [ ]:
# ============================================================
# STEP 2: Enter your Gemini API key
# ============================================================

import getpass
import os

api_key = getpass.getpass("Paste your Gemini API key here: ")
os.environ["GOOGLE_API_KEY"] = api_key
print("✅ API key set!")


In [ ]:
# ============================================================
# STEP 3: Quick test — make sure Gemini is working
# ============================================================

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

response = llm.invoke("Name one approved EGFR inhibitor in one sentence.")
print(response.content)
print("\n✅ Gemini is working!")


---
## Define the Tools

We will build three tools that wrap public, freely accessible databases:

1. **ChEMBL** — EMBL-EBI's database of bioactive molecules and drug-like compounds
2. **ClinicalTrials.gov** — NIH registry of FDA-regulated clinical studies worldwide

Each tool is a Python function decorated with `@tool`. The LLM reads the docstring to decide when to call it.


In [ ]:
# ============================================================
# STEP 4: Tool 1 — Search ChEMBL for a drug or compound
# ============================================================
# ChEMBL REST API: https://www.ebi.ac.uk/chembl/api/data/
# Returns: ChEMBL ID, molecule type, molecular weight, max clinical phase

import requests
from langchain_core.tools import tool


@tool
def search_drug_chembl(drug_name: str) -> str:
    """Search the ChEMBL database for a drug or compound by name.
    Use this tool when the user asks about a specific drug or compound:
    its ChEMBL ID, molecular weight, molecule type, or how far it has
    progressed in clinical development.
    Input: a drug name such as erlotinib, imatinib, osimertinib, or venetoclax.
    Returns: ChEMBL ID, molecule type, molecular weight, and max clinical phase."""

    url = "https://www.ebi.ac.uk/chembl/api/data/molecule"
    params = {
        "pref_name__icontains": drug_name,
        "format": "json",
        "limit": 3,
    }

    response = requests.get(url, params=params, timeout=15)
    if response.status_code != 200:
        return f"Error: ChEMBL API returned status {response.status_code}"

    data = response.json()
    molecules = data.get("molecules", [])
    if not molecules:
        return f"No compounds found in ChEMBL for '{drug_name}'."

    results = []
    for mol in molecules:
        props = mol.get("molecule_properties") or {}
        results.append(
            f"ChEMBL ID: {mol.get('molecule_chembl_id', 'N/A')}\n"
            f"  Preferred name: {mol.get('pref_name', 'N/A')}\n"
            f"  Molecule type: {mol.get('molecule_type', 'N/A')}\n"
            f"  Molecular weight: {props.get('full_mwt', 'N/A')} Da\n"
            f"  Max clinical phase: {mol.get('max_phase', 'N/A')}\n"
            f"  First approval year: {mol.get('first_approval', 'N/A')}\n"
            f"  ChEMBL URL: https://www.ebi.ac.uk/chembl/compound_report_card/{mol.get('molecule_chembl_id', '')}"
        )

    return "\n\n".join(results)


In [ ]:
# ============================================================
# STEP 5: Test Tool 1 independently
# ============================================================
# Always test tools before giving them to the agent.

result = search_drug_chembl.invoke("erlotinib")
print(result)


In [ ]:
# ============================================================
# STEP 6: Tool 2 — Get mechanism of action from ChEMBL
# ============================================================
# Uses the ChEMBL mechanism endpoint.
# Returns: action type, target name, target ChEMBL ID.


@tool
def get_drug_mechanism(drug_name: str) -> str:
    """Look up the mechanism of action for a drug in ChEMBL.
    Use this tool when the user asks HOW a drug works, what its primary
    target is, or what type of pharmacological action it has (inhibitor,
    agonist, antagonist, etc.).
    Input: a drug name such as imatinib, erlotinib, or vemurafenib.
    Returns: mechanism of action, action type, and primary target."""

    # First find the ChEMBL ID for the drug
    search_url = "https://www.ebi.ac.uk/chembl/api/data/molecule"
    search_params = {"pref_name__icontains": drug_name, "format": "json", "limit": 1}
    search_resp = requests.get(search_url, params=search_params, timeout=15)

    if search_resp.status_code != 200:
        return f"Error searching for '{drug_name}': status {search_resp.status_code}"

    mols = search_resp.json().get("molecules", [])
    if not mols:
        return f"No compound found in ChEMBL for '{drug_name}'."

    chembl_id = mols[0]["molecule_chembl_id"]
    drug_label = mols[0].get("pref_name", drug_name)

    # Now fetch mechanism of action
    mech_url = "https://www.ebi.ac.uk/chembl/api/data/mechanism"
    mech_params = {"molecule_chembl_id": chembl_id, "format": "json", "limit": 5}
    mech_resp = requests.get(mech_url, params=mech_params, timeout=15)

    if mech_resp.status_code != 200:
        return f"Error fetching mechanism for {chembl_id}: status {mech_resp.status_code}"

    mechanisms = mech_resp.json().get("mechanisms", [])
    if not mechanisms:
        return f"No mechanism of action data found for {drug_label} ({chembl_id})."

    lines = [f"Mechanisms of action for {drug_label} ({chembl_id}):"]
    for m in mechanisms:
        lines.append(
            f"  • Action: {m.get('action_type', 'N/A')}\n"
            f"    Target: {m.get('target_chembl_id', 'N/A')}\n"
            f"    Mechanism description: {m.get('mechanism_of_action', 'N/A')}\n"
            f"    References: {len(m.get('mechanism_refs', []))} citations"
        )
    return "\n".join(lines)


In [ ]:
# ============================================================
# STEP 7: Test Tool 2 independently
# ============================================================

result = get_drug_mechanism.invoke("imatinib")
print(result)


In [ ]:
# ============================================================
# STEP 8: Tool 3 — Search ClinicalTrials.gov
# ============================================================
# ClinicalTrials.gov v2 API: https://clinicaltrials.gov/api/v2/
# Returns: trial NCT ID, title, phase, status, sponsor, enrollment.


@tool
def search_clinical_trials(query: str) -> str:
    """Search ClinicalTrials.gov for clinical trials matching a drug, target,
    or disease. Use this tool when the user asks about ongoing or completed
    clinical trials, what phase a drug is in, how many trials exist for a
    target, or who is sponsoring trials in a therapeutic area.
    Input: a drug name, gene target, or disease term such as 'osimertinib',
    'KRAS G12C', 'non-small cell lung cancer', or 'CDK4/6 inhibitor'.
    Returns: up to 5 matching trials with phase, status, sponsor, and enrollment."""

    url = "https://clinicaltrials.gov/api/v2/studies"
    params = {
        "query.term": query,
        "pageSize": 5,
        "format": "json",
        "fields": "NCTId,BriefTitle,Phase,OverallStatus,LeadSponsorName,EnrollmentCount,StartDate",
    }

    response = requests.get(url, params=params, timeout=15)
    if response.status_code != 200:
        return f"Error: ClinicalTrials.gov API returned status {response.status_code}"

    data = response.json()
    studies = data.get("studies", [])
    if not studies:
        return f"No clinical trials found for '{query}'."

    total = data.get("totalCount", "unknown")
    lines = [f"Found {total} total trials for '{query}'. Showing top {len(studies)}:\n"]

    for s in studies:
        proto = s.get("protocolSection", {})
        id_mod = proto.get("identificationModule", {})
        status_mod = proto.get("statusModule", {})
        design_mod = proto.get("designModule", {})
        sponsor_mod = proto.get("sponsorCollaboratorsModule", {})

        lines.append(
            f"NCT ID: {id_mod.get('nctId', 'N/A')}\n"
            f"  Title: {id_mod.get('briefTitle', 'N/A')}\n"
            f"  Phase: {', '.join(design_mod.get('phases', ['N/A']))}\n"
            f"  Status: {status_mod.get('overallStatus', 'N/A')}\n"
            f"  Sponsor: {sponsor_mod.get('leadSponsor', {}).get('name', 'N/A')}\n"
            f"  Enrollment: {design_mod.get('enrollmentInfo', {}).get('count', 'N/A')} participants\n"
            f"  Start date: {status_mod.get('startDateStruct', {}).get('date', 'N/A')}\n"
            f"  URL: https://clinicaltrials.gov/study/{id_mod.get('nctId', '')}"
        )

    return "\n".join(lines)


In [ ]:
# ============================================================
# STEP 9: Test Tool 3 independently
# ============================================================

result = search_clinical_trials.invoke("KRAS G12C inhibitor lung cancer")
print(result)


---
## Build the Agent

Now we hand all three tools to the same `create_react_agent` pattern from Notebook 02.
The LLM will autonomously decide which tool(s) to call — and in what order — based on your question.


In [ ]:
# ============================================================
# STEP 10: Create the drug discovery agent
# ============================================================

from langgraph.prebuilt import create_react_agent

tools = [search_drug_chembl, get_drug_mechanism, search_clinical_trials]
agent = create_react_agent(model=llm, tools=tools)

print("✅ Drug discovery agent created with 3 tools:")
print("   • search_drug_chembl      — ChEMBL compound lookup")
print("   • get_drug_mechanism      — ChEMBL mechanism of action")
print("   • search_clinical_trials  — ClinicalTrials.gov search")


In [ ]:
# ============================================================
# STEP 11: Helper function — stream agent reasoning steps
# ============================================================
# This is the same helper as Notebook 02.
# It lets you see WHICH tool the agent called and WHY.


def ask_agent(question):
    print(f"❓ Question: {question}")
    print("=" * 70)

    inputs = {"messages": [{"role": "user", "content": question}]}

    for step in agent.stream(inputs, stream_mode="values"):
        last_msg = step["messages"][-1]

        if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
            for tc in last_msg.tool_calls:
                print(f"\n🔧 Calling tool: {tc['name']}")
                print(f"   Input: {tc['args']}")

        elif last_msg.type == "tool":
            content = last_msg.content
            preview = content[:600] + "..." if len(content) > 600 else content
            print(f"\n📋 Tool result (preview):\n{preview}")

    final = step["messages"][-1].content
    print(f"\n{'=' * 70}")
    print(f"💬 Final Answer:\n{final}")


---
## Demo: Ask the Agent Drug Discovery Questions

Watch how the agent decides which tool(s) to call based on the question.


In [ ]:
# ============================================================
# STEP 12: Demo 1 — Compound lookup (single tool)
# ============================================================

ask_agent("What is osimertinib? Give me its ChEMBL ID, molecular weight, and clinical phase.")


In [ ]:
# ============================================================
# STEP 13: Demo 2 — Mechanism + compound (two tools)
# ============================================================

ask_agent("How does vemurafenib work and what target does it inhibit?")


In [ ]:
# ============================================================
# STEP 14: Demo 3 — Clinical landscape (single tool, broad query)
# ============================================================

ask_agent("What Phase 3 clinical trials are ongoing for CDK4/6 inhibitors in breast cancer?")


In [ ]:
# ============================================================
# STEP 15: Demo 4 — Full pipeline (all three tools chained)
# ============================================================
# The agent must: (1) look up the drug, (2) get its mechanism,
# (3) search for trials — all autonomously.

ask_agent(
    "I'm evaluating selpercatinib as a potential competitor. "
    "What is it, how does it work, and are there any ongoing trials?"
)


---
## What You Built

You created a drug discovery agent that:

1. **Searches ChEMBL** for compound properties and clinical stage
2. **Retrieves mechanisms of action** so you understand how a drug works
3. **Queries ClinicalTrials.gov** for the competitive clinical landscape
4. **Chains these tools autonomously** — the LLM decides the order

The architecture is **identical to Notebook 02**. You added new tools; the agent loop stayed the same.

---

## Next Steps

- **Notebook 05** — Extract structured data (IC50 tables, experimental conditions) from paper abstracts using Pydantic schemas
- **Notebook 06** — Combine multiple specialized agents into a coordinated pipeline that produces a full target assessment report
- **Local deployment** — Replace `ChatGoogleGenerativeAI` with `ChatOllama` (one line) to keep your target names private
